# 🧪 Sesión 23 · Operaciones CRUD Transaccionales y Async Sessions

**Desarrollo de Aplicaciones Web (DAB-2201)** · TecNM Cancún  
**Docente:** MCC. Iván Márquez Larios (`ijmarquezl`)  
**Repositorio:** https://github.com/ijmarquezl/tecnm-desarrollo-web

> **La pregunta de hoy:** ¿Por qué esperar a la base de datos no debe bloquear el servidor?

---

## 🖥️ Así se verá el resultado final

```http
POST /api/libros  {"titulo":"Rayuela","anio":1963}  -> 201
GET  /api/libros                                  -> 200
GET  /api/libros/1                                -> 200
PUT  /api/libros/1  {"titulo":"Rayuela (edición)"} -> 200
DELETE /api/libros/1                              -> 204
GET  /api/libros/1                                -> 404
```

_La base de datos sigue siendo SQLite; lo que cambia es cómo la aplicación espera sus respuestas._

**No hay pasos intermedios.** El cuaderno declara el resultado que debe existir al final; tú decides la estructura, los modelos y el orden de tu implementación. La celda de verificación compara tu resultado con este contrato.

---

## 🎯 Al finalizar podrás

- Implementar un CRUD con SQLAlchemy asíncrono.
- Usar AsyncSession y aiosqlite.
- Verificar 201, 200, 404 y 204 en el mismo flujo.

## ✅ Criterios de aceptación

El verificador ejecutará los casos HTTP y comprobará los archivos del contrato. Todo debe quedar en verde:

- [ ] Crear un libro → HTTP 201
- [ ] Listar los libros → HTTP 200
- [ ] Actualizar el libro → HTTP 200
- [ ] Eliminar el libro → HTTP 204
- [ ] El libro eliminado ya no existe → HTTP 404
- [ ] La persistencia es asíncrona

## 🧩 El reto

Implementa un CRUD completo con SQLAlchemy asíncrono y aiosqlite. El servidor no debe bloquearse mientras espera la base.

**Requisitos del contrato**

- Usa create_async_engine con sqlite+aiosqlite.
- Cada endpoint usa AsyncSession y await.
- POST responde 201; GET 200; PUT 200; DELETE 204.
- Un id inexistente responde 404.
- La tabla se crea al iniciar la aplicación.

> **Pista conceptual:** Si tu endpoint es `def` y usas una sesión síncrona, estás bloqueando el event loop. La regla es: motor async, sesión async, endpoint async.

## 📦 Entregable

Cuaderno en verde y bitácora `auditorias/auditoria_sesion23.md` con la diferencia entre def y async def.


## ⚙️ Preparación del entorno

Ejecuta esta celda una sola vez. Instala las herramientas del track **Python / FastAPI** y descarga el motor de verificación del repositorio. No necesitas instalar nada en tu computadora: todo corre en Colab.


In [1]:
import os, sys
print("⏳ Preparando el entorno de Desarrollo Web...")
!pip install -q fastapi uvicorn httpx pydantic "passlib[bcrypt]" bcrypt python-jose[cryptography] PyJWT "sqlalchemy[asyncio]" greenlet aiosqlite alembic python-multipart
!wget -q -O verificar.py https://raw.githubusercontent.com/ijmarquezl/tecnm-desarrollo-web/c275993ba69ef2a8cf650b2f127b55c8356279d7/laboratorios/verificar.py
if not os.path.exists("verificar.py") or os.path.getsize("verificar.py") < 1000:
    raise RuntimeError("No se pudo descargar verificar.py. Revisa tu conexión o abre el cuaderno desde el repositorio del curso.")
print("✅ Entorno listo")
print("Python:", sys.version.split()[0])
print("Motor de verificación:", os.path.getsize("verificar.py"), "bytes")


⏳ Preparando el entorno de Desarrollo Web...
✅ Entorno listo
Python: 3.13.18
Motor de verificación: 13395 bytes


## 🧩 El reto: escribe el resultado

Crea el o los archivos del contrato. No hay una plantilla de pasos: el esqueleto solo te da el punto de partida. Todo lo que sigue lo decides tú.

**Archivos que debes producir:** `app.py`


### 📄 `app.py`

Archivo principal de la implementación.

In [8]:

%%writefile app.py
from fastapi import FastAPI, HTTPException
from sqlalchemy import Column, Integer, String, create_engine
from sqlalchemy.orm import declarative_base, sessionmaker

app = FastAPI(title="Mi CRUD de libros")


DATABASE_URL = "sqlite:///./biblioteca.db"

engine = create_engine(
    DATABASE_URL,
    connect_args={"check_same_thread": False}
)

Base = declarative_base()
SessionLocal = sessionmaker(bind=engine)

class Libro(Base):
    __tablename__ = "libros"

    id = Column(Integer, primary_key=True, index=True)
    titulo = Column(String, nullable=False)
    anio = Column(Integer, nullable=False)


@app.post("/api/libros", status_code=201)
def crear_libro(datos: dict):
    db = SessionLocal()

    libro = Libro(
        titulo=datos.get("titulo"),
        anio=datos.get("anio")
    )

    db.add(libro)
    db.commit()
    db.refresh(libro)

    return {
        "id": libro.id,
        "titulo": libro.titulo,
        "anio": libro.anio
    }

@app.get("/api/libros")
def listar_libros():
    db = SessionLocal()
    libros = db.query(Libro).all()

    return {
        "libros": [
            {"id": libro.id, "titulo": libro.titulo}
            for libro in libros
        ]
    }

@app.get("/api/libros/{libro_id}")
def obtener_libro(libro_id: int):
    db = SessionLocal()

    libro = db.query(Libro).filter(
        Libro.id == libro_id
    ).first()

    if libro is None:
        raise HTTPException(404, "Libro no encontrado")

    return {
        "id": libro.id,
        "titulo": libro.titulo,
        "anio": libro.anio
    }

@app.put("/api/libros/{libro_id}")
def actualizar_libro(libro_id: int, datos: dict):
    db = SessionLocal()

    libro = db.query(Libro).filter(
        Libro.id == libro_id
    ).first()

    if libro is None:
        raise HTTPException(404, "Libro no encontrado")

    libro.titulo = datos.get("titulo", libro.titulo)
    libro.anio = datos.get("anio", libro.anio)

    db.commit()
    db.refresh(libro)

    return {
        "id": libro.id,
        "titulo": libro.titulo,
        "anio": libro.anio
    }

@app.delete("/api/libros/{libro_id}", status_code=204)
def eliminar_libro(libro_id: int):
    db = SessionLocal()

    libro = db.query(Libro).filter(
        Libro.id == libro_id
    ).first()

    if libro is None:
        raise HTTPException(404, "Libro no encontrado")

    db.delete(libro)
    db.commit()

    return None



Overwriting app.py


In [12]:


%%writefile app.py
from fastapi import FastAPI, HTTPException, Response
from pydantic import BaseModel, ConfigDict
from sqlalchemy import Integer, String, select
from sqlalchemy.ext.asyncio import (
    AsyncSession,
    async_sessionmaker,
    create_async_engine
)
from sqlalchemy.orm import DeclarativeBase, Mapped, mapped_column

# 1. Conexion asincrona con SQLite
DATABASE_URL = "sqlite+aiosqlite:///./biblioteca.db"

engine = create_async_engine(
    DATABASE_URL,
    echo=False
)

SessionLocal = async_sessionmaker(
    bind=engine,
    class_=AsyncSession,
    expire_on_commit=False
)

# 2. Modelo de base de datos
class Base(DeclarativeBase):
    pass


class Libro(Base):
    __tablename__ = "libros"

    id: Mapped[int] = mapped_column(
        Integer,
        primary_key=True,
        index=True
    )

    titulo: Mapped[str] = mapped_column(
        String,
        nullable=False
    )

    anio: Mapped[int] = mapped_column(
        Integer,
        nullable=False
    )


# 3. Modelos de entrada y salida
class LibroEntrada(BaseModel):
    titulo: str
    anio: int


class LibroSalida(BaseModel):
    model_config = ConfigDict(from_attributes=True)

    id: int
    titulo: str
    anio: int


# 4. Crear la aplicacion
app = FastAPI(title="CRUD asincrono de libros")

tablas_creadas = False


# 5. Preparar la base de datos antes de atender peticiones
@app.middleware("http")
async def preparar_base_datos(request, call_next):
    global tablas_creadas

    if not tablas_creadas:
        async with engine.begin() as conn:
            await conn.run_sync(Base.metadata.create_all)

        tablas_creadas = True

    response = await call_next(request)
    return response


# 6. POST: Crear libro
@app.post(
    "/api/libros",
    response_model=LibroSalida,
    status_code=201
)
async def crear_libro(datos: LibroEntrada):

    async with SessionLocal() as session:

        libro = Libro(
            titulo=datos.titulo,
            anio=datos.anio
        )

        session.add(libro)

        await session.commit()
        await session.refresh(libro)

        return libro


# 7. GET: Listar libros
@app.get("/api/libros")
async def listar_libros():

    async with SessionLocal() as session:

        resultado = await session.execute(
            select(Libro).order_by(Libro.id)
        )

        libros = resultado.scalars().all()

        return {"libros": libros}


# 8. GET: Consultar libro por ID
@app.get(
    "/api/libros/{libro_id}",
    response_model=LibroSalida
)
async def obtener_libro(libro_id: int):

    async with SessionLocal() as session:

        libro = await session.get(Libro, libro_id)

        if libro is None:
            raise HTTPException(
                status_code=404,
                detail="Libro no encontrado"
            )

        return libro


# 9. PUT: Actualizar libro
@app.put(
    "/api/libros/{libro_id}",
    response_model=LibroSalida
)
async def actualizar_libro(
    libro_id: int,
    datos: LibroEntrada
):

    async with SessionLocal() as session:

        libro = await session.get(Libro, libro_id)

        if libro is None:
            raise HTTPException(
                status_code=404,
                detail="Libro no encontrado"
            )

        libro.titulo = datos.titulo
        libro.anio = datos.anio

        await session.commit()
        await session.refresh(libro)

        return libro


# 10. DELETE: Eliminar libro
@app.delete(
    "/api/libros/{libro_id}",
    status_code=204
)
async def eliminar_libro(libro_id: int):

    async with SessionLocal() as session:

        libro = await session.get(Libro, libro_id)

        if libro is None:
            raise HTTPException(
                status_code=404,
                detail="Libro no encontrado"
            )

        await session.delete(libro)
        await session.commit()

    return Response(status_code=204)




Overwriting app.py


## 🧪 Tu contrato C-R-E-O

Antes de pedir ayuda a la IA, escribe el contrato que describe el resultado. Esto alimenta tu bitácora y te obliga a pensar como arquitecto:

| Elemento | Tu definición |
| :--- | :--- |
| **Contexto** | Estoy trabajando con Python, FastAPI y SQLAlchemy en Google Colab. Utilizo SQLite como base de datos y las herramientas aiosqlite y AsyncSession para realizar operaciones asíncronas . |
| **Rol** | La IA debe actuar como un desarrollador especializado en Python y FastAPI, ayudándome a implementar un CRUD asíncrono y explicando los errores que puedan aparecer. |
| **Especificación** | Necesito implementar las operaciones para crear, consultar, listar, actualizar y eliminar libros. El sistema debe utilizar create_async_engine, AsyncSession, async def y await. Debe crear la tabla al iniciar, responder con HTTP 201 al crear, 200 al consultar y actualizar, 204 al eliminar y 404 cuando un libro no exista. |
| **Output** | Espero recibir el código completo de app.py, organizado y comentado, utilizando SQLite con aiosqlite. También debe funcionar con el verificador oficial y superar los cinco casos de prueba. |

> No pegues código de la IA sin auditarlo. La responsabilidad técnica del resultado es tuya.


## 🔍 Verificación del resultado

Ejecuta esta celda cuando tu implementación esté lista. La verificación no modifica tu código: solo observa el comportamiento y compara con el contrato. Si algo falla, el mensaje te dice **qué esperaba** y **qué recibió**.


In [13]:
from verificar import probar_http

casos = [{'descripcion': 'Crear un libro',
  'metodo': 'POST',
  'ruta': '/api/libros',
  'json': {'titulo': 'Rayuela', 'anio': 1963},
  'estado': 201,
  'json_contiene': {'id': 1, 'titulo': 'Rayuela'}},
 {'descripcion': 'Listar los libros',
  'metodo': 'GET',
  'ruta': '/api/libros',
  'estado': 200,
  'json_contiene': {'libros': [{'id': 1}]}},
 {'descripcion': 'Actualizar el libro',
  'metodo': 'PUT',
  'ruta': '/api/libros/1',
  'json': {'titulo': 'Rayuela (edición crítica)', 'anio': 1963},
  'estado': 200,
  'json_contiene': {'id': 1, 'titulo': 'Rayuela (edición crítica)'}},
 {'descripcion': 'Eliminar el libro', 'metodo': 'DELETE', 'ruta': '/api/libros/1', 'estado': 204},
 {'descripcion': 'El libro eliminado ya no existe', 'metodo': 'GET', 'ruta': '/api/libros/1', 'estado': 404}]

probar_http(casos, archivo="app.py", limpiar=['biblioteca.db'], env={})


VERIFICACIÓN DEL RESULTADO · 5 casos · app.py
✅ 01. Crear un libro
✅ 02. Listar los libros
✅ 03. Actualizar el libro
✅ 04. Eliminar el libro
✅ 05. El libro eliminado ya no existe
------------------------------------------------------------------------------
RESULTADO: 5/5 casos correctos. ¡Resultado logrado!


In [14]:
from verificar import revisar_archivo, resumen

resultados = []
resultados.append(revisar_archivo("app.py", regex=["AsyncSession|async_sessionmaker|aiosqlite"], descripcion="La persistencia es asíncrona"))
resumen(resultados)


✅ La persistencia es asíncrona
------------------------------------------------------------------------------
RESULTADO: 1/1 comprobaciones correctas. ¡Resultado logrado!


## 📝 Cierre y bitácora

1. Copia la plantilla `plantillas/auditoria_sesion23.md` a `auditorias/auditoria_sesion23.md` en tu fork.
2. Responde con tus palabras:
   - ¿Qué resultado te costó más alcanzar y por qué?

   Lo que más me costó fue conseguir que todas las operaciones CRUD funcionaran correctamente de manera asíncrona. Al principio, aunque el código parecía estar bien, el verificador mostraba errores HTTP 500. Tuve que revisar cómo se creaba la base de datos y cómo se ejecutaban las operaciones con AsyncSession y await.

   - ¿Qué decisión de diseño tomaste que no estaba escrita en el contrato?

   Decidí preparar la base de datos antes de atender las solicitudes, utilizando un middleware para comprobar que las tablas estuvieran creadas. Esto lo agregué porque necesitaba asegurar que el sistema pudiera funcionar correctamente incluso cuando el verificador realizara las pruebas.

   - ¿Qué error detectó la verificación que tú no habías previsto?

   El error que no había previsto fue que las pruebas devolvían HTTP 500 porque las tablas de la base de datos todavía no estaban creadas cuando se realizaban las solicitudes. Esto me ayudó a entender que no basta con que las rutas estén programadas; también es necesario comprobar que la conexión y la inicialización de la base de datos funcionen correctamente.
   
3. Haz commit de tu cuaderno ejecutado y de tu bitácora.

**La sesión termina cuando el resultado está en verde, no cuando el código "parece funcionar".**
